# Brute-Force Pairs Finder
This notebook implements the "brute-force" approach to finding tradable pairs. The process is:

Define a Universe: We'll select a group of related stocks (e.g., the S&P 500 Financials) to test.
Get Data: We'll use our DataHandler to fetch 3-5 years of daily price data for this universe.
Pair Up: We'll programmatically create every possible unique pair from this universe.
Test for Cointegration: For each pair, we will:
Run an OLS regression to find the hedge ratio (k).
Calculate the "spread" (the residuals from the regression).
Run the Augmented Dickey-Fuller (ADF) test on the spread to see if it's stationary.
Save Results: We'll save all pairs that pass the test (p-value < 0.05) to a list.

In [1]:
import pandas as pd
import numpy as np
import os
import itertools
from dotenv import load_dotenv

# --- Math & Stats Libraries ---
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

# --- Alpaca & Data Handling ---
# We'll use the DataHandler class we built, assuming it's in the 'src' folder
# If running this from the 'notebooks' folder, we need to adjust the path
import sys
sys.path.append('../src') # This allows us to import from the 'src' folder
from data_handler import DataHandler

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# --- Step 1: Define Universe & Timeframe ---

# A targeted "brute-force" is better than testing thousands of stocks.
# Let's use a list of major US financial stocks (components of the XLF ETF).
# This provides a good fundamental reason for cointegration.
UNIVERSE = [
 'BLK', 'SPGI', 'AXP','JPM'
    'COF', 'USB','MCO','PYPL','V','MA','BAC','C','WFC','GS','MS','SCHW','PNC','TFC','FITB','KEY','CFG','HBAN','ZION','RF','CMA','DOV']

# Cointegration is a long-term relationship, so we need several years of data.
START_DATE = '2019-01-01'
END_DATE = '2022-01-01' # Test on data up to the start of 2024
TIMEFRAME = '1D' # Daily data is standard for this

print(f"Testing {len(UNIVERSE)} stocks from {START_DATE} to {END_DATE}.")

Testing 25 stocks from 2019-01-01 to 2022-01-01.


In [3]:
# --- Step 2: Get & Prepare Price Data ---

print("Fetching historical data...")
dh = DataHandler(paper_trading=True)

# Get the dictionary of DataFrames from our handler
hist_data = dh.get_historical_bars(UNIVERSE, TIMEFRAME, START_DATE, END_DATE)

# We need to combine this into a single DataFrame of closing prices
print("Processing data into a master DataFrame...")
close_prices = pd.DataFrame()

for symbol in UNIVERSE:
    if hist_data and symbol in hist_data and not hist_data[symbol].empty:
        # We need to make sure the index is a DatetimeIndex
        df = hist_data[symbol]
        df.index = pd.to_datetime(df.index)
        
        # Get just the closing prices
        close_prices[symbol] = df['close']
    else:
        print(f"Warning: No data for {symbol}. It will be skipped.")

# Drop any rows with missing data for any stock
close_prices.dropna(inplace=True)

# Re-update our universe list to only include stocks we have data for
UNIVERSE = close_prices.columns.tolist()

print(f"Data prepared. Master DataFrame shape: {close_prices.shape}")
display(close_prices.head())

Fetching historical data...
Data Handler (alpaca-py) initialized.
Submitting data request to Alpaca...
Processing data into a master DataFrame...
Data prepared. Master DataFrame shape: (757, 24)
Processing data into a master DataFrame...
Data prepared. Master DataFrame shape: (757, 24)


,BLK,SPGI,AXP,USB,MCO,PYPL,V,MA,BAC,C,...,PNC,TFC,FITB,KEY,CFG,HBAN,ZION,RF,CMA,DOV
timestamp,,,,,,,,,,,,,,,,,,,,,
2019-01-02 05:00:00+00:00,389.42,169.85,95.68,46.35,140.85,85.75,132.92,189.74,24.96,53.53,...,118.81,44.39,24.20,15.02,30.51,12.14,41.58,13.67,69.74,71.25
2019-01-03 05:00:00+00:00,377.98,164.37,93.43,45.70,136.18,82.09,128.13,181.18,24.56,52.56,...,118.27,44.09,24.18,15.07,30.37,12.04,41.35,13.65,69.28,69.79
2019-01-04 05:00:00+00:00,391.82,172.26,97.64,46.83,143.28,86.27,133.65,189.76,25.58,55.13,...,121.23,45.25,24.76,15.58,31.62,12.38,42.62,14.15,71.50,72.96
2019-01-07 05:00:00+00:00,392.91,173.64,98.17,46.61,143.81,86.93,136.06,191.22,25.56,55.61,...,120.96,45.61,24.90,15.74,32.01,12.43,42.79,14.39,72.13,73.93
2019-01-08 05:00:00+00:00,397.91,175.23,98.65,46.91,145.90,88.70,136.80,192.28,25.51,55.46,...,121.15,45.72,25.03,15.73,32.01,12.55,43.18,14.54,72.32,75.56


In [4]:
# --- Step 3: Cointegration Test Function ---

def find_cointegration(series_1, series_2):
    """
    Tests for cointegration between two price series.
    
    1. Runs OLS regression: series_1 = k * series_2 + intercept
    2. Calculates the spread (residuals).
    3. Runs ADF test on the spread.
    
    :return: (hedge_ratio, adf_p_value)
    """
    
    # 1. Run OLS regression
    # We add a constant (intercept) to the independent variable
    series_2_with_const = sm.add_constant(series_2)
    model = sm.OLS(series_1, series_2_with_const)
    results = model.fit()
    
    hedge_ratio = results.params[1] # 'k'
    
    # 2. Calculate the spread
    spread = series_1 - hedge_ratio * series_2
    
    # 3. Run ADF test on the spread
    # The null hypothesis of ADF is that the series IS non-stationary
    # We want a low p-value to reject the null hypothesis
    adf_test = adfuller(spread)
    adf_p_value = adf_test[1] # The p-value
    
    return hedge_ratio, adf_p_value

In [5]:
# --- Step 4: The Brute-Force Loop ---

print("Running cointegration tests on all pairs...")

# Set our significance threshold
P_VALUE_THRESHOLD = 0.05

# Create all unique pairs of stocks
all_pairs = list(itertools.combinations(UNIVERSE, 2))

cointegrated_pairs = []

for symbol_a, symbol_b in all_pairs:
    
    series_a = close_prices[symbol_a]
    series_b = close_prices[symbol_b]
    
    hedge_ratio, p_value = find_cointegration(series_a, series_b)
    
    if p_value < P_VALUE_THRESHOLD:
        print(f"Found Cointegrated Pair: {symbol_a} / {symbol_b} | p-value: {p_value:.4f} | hedge_ratio: {hedge_ratio:.2f}")
        cointegrated_pairs.append({
            'symbol_a': symbol_a,
            'symbol_b': symbol_b,
            'hedge_ratio': hedge_ratio,
            'p_value': p_value
        })

print("\n--- Test Complete ---")
print(f"Total pairs tested: {len(all_pairs)}")
print(f"Total cointegrated pairs found: {len(cointegrated_pairs)}")

Running cointegration tests on all pairs...


C:\Users\trash\AppData\Local\Temp\ipykernel_28196\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = results.params[1] # 'k'
C:\Users\trash\AppData\Local\Temp\ipykernel_28196\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = results.params[1] # 'k'
C:\Users\trash\AppData\Local\Temp\ipykernel_28196\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = resu

Found Cointegrated Pair: BLK / MS | p-value: 0.0430 | hedge_ratio: 7.91


C:\Users\trash\AppData\Local\Temp\ipykernel_28196\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = results.params[1] # 'k'
C:\Users\trash\AppData\Local\Temp\ipykernel_28196\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = results.params[1] # 'k'
C:\Users\trash\AppData\Local\Temp\ipykernel_28196\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = resu

Found Cointegrated Pair: AXP / BAC | p-value: 0.0326 | hedge_ratio: 3.60
Found Cointegrated Pair: AXP / PNC | p-value: 0.0114 | hedge_ratio: 0.78
Found Cointegrated Pair: AXP / FITB | p-value: 0.0138 | hedge_ratio: 3.19


...

C:\Users\trash\AppData\Local\Temp\ipykernel_28196\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = results.params[1] # 'k'
C:\Users\trash\AppData\Local\Temp\ipykernel_28196\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = results.params[1] # 'k'
C:\Users\trash\AppData\Local\Temp\ipykernel_28196\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = resu

Found Cointegrated Pair: TFC / KEY | p-value: 0.0006 | hedge_ratio: 2.22
Found Cointegrated Pair: TFC / CFG | p-value: 0.0315 | hedge_ratio: 0.98
Found Cointegrated Pair: TFC / HBAN | p-value: 0.0019 | hedge_ratio: 3.34
Found Cointegrated Pair: TFC / ZION | p-value: 0.0313 | hedge_ratio: 0.78


C:\Users\trash\AppData\Local\Temp\ipykernel_28196\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = results.params[1] # 'k'
C:\Users\trash\AppData\Local\Temp\ipykernel_28196\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = results.params[1] # 'k'
C:\Users\trash\AppData\Local\Temp\ipykernel_28196\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = resu

Found Cointegrated Pair: KEY / CFG | p-value: 0.0171 | hedge_ratio: 0.44
Found Cointegrated Pair: CFG / RF | p-value: 0.0382 | hedge_ratio: 2.05

--- Test Complete ---
Total pairs tested: 276
Total cointegrated pairs found: 18

--- Test Complete ---
Total pairs tested: 276
Total cointegrated pairs found: 18


C:\Users\trash\AppData\Local\Temp\ipykernel_28196\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = results.params[1] # 'k'
C:\Users\trash\AppData\Local\Temp\ipykernel_28196\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = results.params[1] # 'k'
C:\Users\trash\AppData\Local\Temp\ipykernel_28196\1384126882.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hedge_ratio = resu

In [6]:
# --- Step 5: Analyze Results ---

# Convert the results into a clean DataFrame for analysis
results_df = pd.DataFrame(cointegrated_pairs)

print("\nCointegrated Pairs Summary:")

display(results_df)

# You can now save this to a file
# results_df.to_csv('cointegrated_pairs.csv', index=False)


Cointegrated Pairs Summary:


,symbol_a,symbol_b,hedge_ratio,p_value
0,BLK,MS,7.911508,0.042965
1,AXP,BAC,3.596525,0.032571
2,AXP,PNC,0.779144,0.011383
3,AXP,FITB,3.193734,0.013758
4,V,MA,0.535062,0.042191
5,V,GS,0.267517,0.045588
6,V,MS,0.993562,0.037482
7,MA,MS,1.740668,0.045269
8,BAC,PNC,0.211015,0.004645
9,BAC,FITB,0.871249,0.036462


In [7]:
from pathlib import Path
OUTPUT_DIR = (Path.cwd().parent / 'data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / 'cointegrated_pairs.csv'
if 'results_df' not in globals():
    raise NameError("results_df is not defined. Run the cell that builds the summary DataFrame first.")
results_df.to_csv(output_path, index=False)
print(f"Saved {len(results_df)} cointegrated pairs to {output_path}")

Saved 18 cointegrated pairs to c:\Users\trash\trading-bot\data\cointegrated_pairs.csv
